# NIBP BP-Source Labeling (Pre-Stage 3)

**Goal**: For the 108 patients who have no invasive ART_MBP track, estimate MAP from
non-invasive NIBP cuff readings (Solar8000/NIBP_SBP and Solar8000/NIBP_DBP).
Decide which episodes are usable, flag any imputed ones, and produce four output
files that Stage 3 (outcome labeling) will build on.

**Output files** (all saved into `NIBP_data/`):
1. `arrhythmia_episodes_updated.csv` — included episodes with `bp_source` + `nibp_outcome_imputed`
2. `clinical_lab_updated.csv` — clinical rows for included patients, with `bp_source`
3. `arrhythmia_episodes_summary.csv` — missingness summary for episode columns
4. `clinical_lab_summary.csv` — missingness summary for clinical columns

In [1]:
# Standard libraries we need for this notebook.
import numpy as np        # Fast array math (loading waveform data)
import pandas as pd       # DataFrames (loading / saving CSVs)
import vitaldb            # VitalDB Python client (loads waveform tracks per patient)
from pathlib import Path  # Cross-platform file paths
import random             # For reproducibly sampling 3 episodes in Verification 3

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [ ]:
# This notebook lives in notebooks/, so going up one level (..) reaches the project root.
BASE_DIR     = Path('..').resolve()
DATA_INTERIM = BASE_DIR / 'data' / 'interim'
OUTPUT_DIR   = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EPISODES_CSV = DATA_INTERIM / 'arrhythmia_episodes_482.csv'
CLINICAL_CSV = DATA_INTERIM / 'imputed_477_cases.csv'

# VitalDB track names
ART_MBP_TRACK  = 'Solar8000/ART_MBP'   # Invasive arterial MAP (gold standard)
NIBP_SBP_TRACK = 'Solar8000/NIBP_SBP'  # Non-invasive cuff systolic BP
NIBP_DBP_TRACK = 'Solar8000/NIBP_DBP'  # Non-invasive cuff diastolic BP

# We resample every track to 1-second intervals.
# Index t in the returned array = second t from the recording start.
INTERVAL_SEC = 1.0

# Artifact-cleaning thresholds (physiological plausibility)
SBP_MIN, SBP_MAX = 60, 220    # mmHg — outside this range, discard the SBP reading
DBP_MIN, DBP_MAX = 30, 130    # mmHg — outside this range, discard the DBP reading
MAP_MIN, MAP_MAX = 40, 150    # mmHg — computed MAP still implausible? also discard

# Time window constants
OUTCOME_WINDOW_SEC  = 300   # seconds, primary window after episode start

EXPANDED_BEFORE_SEC = 300   # seconds before episode_start_sec
EXPANDED_AFTER_SEC  = 600   # seconds after  episode_start_sec

print('Paths and constants set.')
print(f'  Episode input : {EPISODES_CSV}')
print(f'  Clinical input: {CLINICAL_CSV}')
print(f'  Output folder : {OUTPUT_DIR.resolve()}')

In [3]:
# Load the two main input files created in earlier stages.
episodes = pd.read_csv(EPISODES_CSV)   # 1295 arrhythmia episodes (long format)
clinical = pd.read_csv(CLINICAL_CSV)   # 477 patients, clinical + lab features

print(f'Episodes loaded : {len(episodes)} rows, {episodes["caseid"].nunique()} unique patients')
print(f'Clinical loaded : {len(clinical)} rows, {clinical["caseid"].nunique()} unique patients')
print(f'\nEpisode columns : {episodes.columns.tolist()}')

Episodes loaded : 1295 rows, 460 unique patients
Clinical loaded : 477 rows, 477 unique patients

Episode columns : ['caseid', 'episode_number', 'episode_start_sec', 'episode_end_sec', 'episode_duration_sec', 'episode_beat_count', 'episode_dominant_rhythm', 'episode_beat_type', 'episode_rr_cv']


In [4]:
# Ask VitalDB which signal tracks were recorded for each of our 477 patients.
# This requires an internet connection and takes a few seconds.
#
# get_track_names() returns a DataFrame with two columns:
#   caseid  — patient identifier
#   tnames  — Python list of track name strings for that patient

case_ids = clinical['caseid'].tolist()
print(f'Querying VitalDB track availability for {len(case_ids)} patients ...')

track_names_df = vitaldb.get_track_names(caseids=case_ids)

# Check for each patient: does their tnames list include the ART_MBP track?
has_art_mbp = track_names_df['tnames'].apply(lambda names: ART_MBP_TRACK in names)

art_mbp_caseids  = set(track_names_df.loc[ has_art_mbp, 'caseid'].tolist())
nibp_only_caseids = set(track_names_df.loc[~has_art_mbp, 'caseid'].tolist())

print(f'\nPatients WITH  {ART_MBP_TRACK}: {len(art_mbp_caseids)}')
print(f'Patients WITHOUT {ART_MBP_TRACK}: {len(nibp_only_caseids)}')
print(f'  --> These {len(nibp_only_caseids)} patients will use estimated MAP from NIBP_SBP / NIBP_DBP')

Querying VitalDB track availability for 477 patients ...


C:\Users\sukka\AppData\Local\Temp\ipykernel_18476\1753755442.py:11: DeprecationWarning: get_track_names() relies on the per-track 'trks' index, which is deprecated. Track names are available from the .vital file via VitalFile(caseid).get_track_names(). The trks index may be removed in a future release.
  track_names_df = vitaldb.get_track_names(caseids=case_ids)



Patients WITH  Solar8000/ART_MBP: 369
Patients WITHOUT Solar8000/ART_MBP: 108
  --> These 108 patients will use estimated MAP from NIBP_SBP / NIBP_DBP


In [5]:
# Divide the 1295 episodes into two groups based on BP source availability.

art_episodes      = episodes[episodes['caseid'].isin(art_mbp_caseids)].copy()
nibp_all_episodes = episodes[episodes['caseid'].isin(nibp_only_caseids)].copy()

print(f'ART_MBP episodes  : {len(art_episodes)}')
print(f'NIBP-only episodes: {len(nibp_all_episodes)}')
print(f'Total check       : {len(art_episodes) + len(nibp_all_episodes)} (should equal {len(episodes)})')

# Tag ART_MBP episodes right away — they are all automatically included.
# No waveform loading or window checking is needed for these patients;
# Stage 3 will load their ART_MBP track directly.
art_episodes['bp_source']            = 'ART_MBP'
art_episodes['nibp_outcome_imputed'] = False

print('\nART_MBP episodes tagged: bp_source="ART_MBP", nibp_outcome_imputed=False')

ART_MBP episodes  : 982
NIBP-only episodes: 313
Total check       : 1295 (should equal 1295)

ART_MBP episodes tagged: bp_source="ART_MBP", nibp_outcome_imputed=False


In [6]:
# ── Main loop: process the 108 NIBP-only patients ──────────────────────────
#
# For each patient we:
#   1. Load Solar8000/NIBP_SBP and Solar8000/NIBP_DBP at 1-second intervals.
#   2. Apply artifact cleaning (remove physiologically impossible values).
#   3. Compute estimated MAP = DBP + (SBP - DBP) / 3 at each valid second.
#   4. For each episode, check whether any valid MAP estimate falls in the
#      primary 5-minute outcome window [episode_start_sec, +300 s].
#      - If YES  → include, nibp_outcome_imputed = False
#      - If NO   → expand to [episode_start_sec - 300 s, +600 s]
#        - If any valid reading in expanded window → include, nibp_outcome_imputed = True
#        - If still none → exclude episode entirely

# We use groupby('caseid') so we load each patient's waveform ONCE,
# then loop through all that patient's episodes using the same loaded data.

included_nibp_rows = []   # Will become a DataFrame of included NIBP episodes
excluded_nibp_rows = []   # Tracks excluded (caseid, episode_number) pairs

n_patients  = nibp_all_episodes['caseid'].nunique()
patient_idx = 0

for caseid, group in nibp_all_episodes.groupby('caseid'):
    patient_idx += 1
    print(f'[{patient_idx}/{n_patients}] caseid {caseid}: {len(group)} episode(s) ...')

    # ── Step 1: Load NIBP_SBP and NIBP_DBP waveforms ──
    # load_case() returns a 2-D numpy array shaped (n_seconds, n_tracks).
    # Column 0 = NIBP_SBP, column 1 = NIBP_DBP.
    # Seconds where the monitor had no reading are filled with NaN.
    arr = vitaldb.load_case(caseid, [NIBP_SBP_TRACK, NIBP_DBP_TRACK], INTERVAL_SEC)

    # Guard against the rare case where load_case returns nothing.
    if arr is None or len(arr) == 0:
        print(f'  WARNING: no waveform data returned for caseid {caseid}; skipping all episodes.')
        for _, ep in group.iterrows():
            excluded_nibp_rows.append({'caseid': caseid, 'episode_number': int(ep['episode_number'])})
        continue

    sbp = arr[:, 0].copy()   # Systolic BP time series (mmHg)
    dbp = arr[:, 1].copy()   # Diastolic BP time series (mmHg)
    n_total_sec = len(sbp)   # Total recording length in seconds

    # ── Step 2: Artifact cleaning ──
    # Each cleaning step sets bad values to NaN so they are ignored later.

    # 2a. SBP must be in [60, 220] mmHg; outside that range → NaN
    sbp[(sbp < SBP_MIN) | (sbp > SBP_MAX)] = np.nan

    # 2b. DBP must be in [30, 130] mmHg; outside that range → NaN
    dbp[(dbp < DBP_MIN) | (dbp > DBP_MAX)] = np.nan

    # 2c. Rows where SBP ≤ DBP are physically impossible (pulse pressure ≤ 0).
    #     NaN both so the formula doesn't produce a nonsensical MAP.
    bad_relationship = sbp <= dbp
    sbp[bad_relationship] = np.nan
    dbp[bad_relationship] = np.nan

    # ── Step 3: Compute estimated MAP ──
    # Standard formula: MAP ≈ DBP + (pulse_pressure / 3) = DBP + (SBP - DBP) / 3
    # When SBP or DBP is NaN, NumPy propagates NaN automatically.
    estimated_map = dbp + (sbp - dbp) / 3

    # 2d. Even after SBP/DBP cleaning, MAP outside [40, 150] is still implausible.
    estimated_map[(estimated_map < MAP_MIN) | (estimated_map > MAP_MAX)] = np.nan

    # ── Step 4: Check each episode's outcome windows ──
    for _, ep in group.iterrows():
        # episode_start_sec can be fractional; convert to int (floor) for array indexing.
        start_idx = int(ep['episode_start_sec'])

        # --- Primary 5-minute window: [start_idx, start_idx + 300) ---
        win_start = max(0, start_idx)
        win_end   = min(start_idx + OUTCOME_WINDOW_SEC, n_total_sec)

        # Count valid (non-NaN) estimated MAP readings in the primary window.
        if win_start < n_total_sec:
            primary_vals   = estimated_map[win_start:win_end]
            n_valid_primary = int(np.sum(~np.isnan(primary_vals)))
        else:
            # Episode starts after the recording ended — no data possible.
            n_valid_primary = 0

        if n_valid_primary > 0:
            # ✓ At least one valid reading in the primary window → normal inclusion.
            included_nibp_rows.append({
                **ep.to_dict(),
                'bp_source':            'NIBP_estimated',
                'nibp_outcome_imputed': False,
            })

        else:
            # No readings in the primary window — try the expanded window.
            # Expanded: [start_idx - 300, start_idx + 600]
            exp_start = max(0, start_idx - EXPANDED_BEFORE_SEC)
            exp_end   = min(start_idx + EXPANDED_AFTER_SEC, n_total_sec)

            if exp_start < n_total_sec:
                expanded_vals    = estimated_map[exp_start:exp_end]
                n_valid_expanded = int(np.sum(~np.isnan(expanded_vals)))
            else:
                n_valid_expanded = 0

            if n_valid_expanded > 0:
                # ✓ Found a reading in the expanded window → include, flag as imputed.
                included_nibp_rows.append({
                    **ep.to_dict(),
                    'bp_source':            'NIBP_estimated',
                    'nibp_outcome_imputed': True,
                })
            else:
                # ✗ No valid readings anywhere → exclude this episode entirely.
                print(f'  EXCLUDED: episode {int(ep["episode_number"])} '
                      f'(start={ep["episode_start_sec"]:.1f}s) — '
                      f'no valid NIBP readings in either window')
                excluded_nibp_rows.append({
                    'caseid':         caseid,
                    'episode_number': int(ep['episode_number']),
                })

print(f'\nNIBP processing complete.')
print(f'  NIBP episodes included: {len(included_nibp_rows)}')
print(f'  NIBP episodes excluded: {len(excluded_nibp_rows)}')

[1/99] caseid 42: 1 episode(s) ...
[2/99] caseid 158: 3 episode(s) ...
  EXCLUDED: episode 1 (start=18058.4s) — no valid NIBP readings in either window
  EXCLUDED: episode 2 (start=18121.1s) — no valid NIBP readings in either window
  EXCLUDED: episode 3 (start=18231.0s) — no valid NIBP readings in either window
[3/99] caseid 174: 2 episode(s) ...
[4/99] caseid 212: 2 episode(s) ...
[5/99] caseid 253: 2 episode(s) ...
[6/99] caseid 257: 3 episode(s) ...
[7/99] caseid 285: 3 episode(s) ...
[8/99] caseid 365: 1 episode(s) ...
[9/99] caseid 539: 2 episode(s) ...
[10/99] caseid 554: 1 episode(s) ...
[11/99] caseid 569: 8 episode(s) ...
[12/99] caseid 581: 3 episode(s) ...
[13/99] caseid 643: 6 episode(s) ...
[14/99] caseid 696: 2 episode(s) ...
[15/99] caseid 708: 4 episode(s) ...
[16/99] caseid 713: 1 episode(s) ...
[17/99] caseid 862: 2 episode(s) ...
[18/99] caseid 884: 1 episode(s) ...
[19/99] caseid 1072: 3 episode(s) ...
[20/99] caseid 1110: 3 episode(s) ...
[21/99] caseid 1157: 7 ep

In [7]:
# Combine ART_MBP episodes and included NIBP episodes into a single DataFrame.

# Convert the list of dicts (one per included NIBP episode) into a DataFrame.
nibp_included_df = pd.DataFrame(included_nibp_rows)

# Stack the two groups vertically. ignore_index=True gives a clean 0-based index.
all_included_episodes = pd.concat(
    [art_episodes, nibp_included_df],
    ignore_index=True
)

# Sort by caseid then episode number so the file reads logically.
all_included_episodes = (
    all_included_episodes
    .sort_values(['caseid', 'episode_number'])
    .reset_index(drop=True)
)

print(f'Total included episodes : {len(all_included_episodes)}')
print(f'  ART_MBP              : {len(art_episodes)}')
print(f'  NIBP included        : {len(nibp_included_df)}')
print(f'  NIBP excluded        : {len(excluded_nibp_rows)}')
print(f'\nColumns: {all_included_episodes.columns.tolist()}')

Total included episodes : 1284
  ART_MBP              : 982
  NIBP included        : 302
  NIBP excluded        : 11

Columns: ['caseid', 'episode_number', 'episode_start_sec', 'episode_end_sec', 'episode_duration_sec', 'episode_beat_count', 'episode_dominant_rhythm', 'episode_beat_type', 'episode_rr_cv', 'bp_source', 'nibp_outcome_imputed']


In [8]:
# ── File 1: arrhythmia_episodes_updated.csv ────────────────────────────────
# The filtered episode list that Stage 3 will use as its starting point.
# Contains all original episode columns plus bp_source and nibp_outcome_imputed.

output_path_1 = OUTPUT_DIR / 'arrhythmia_episodes_updated.csv'
all_included_episodes.to_csv(output_path_1, index=False)

print(f'File 1 saved: {output_path_1}')
print(f'  Rows    : {len(all_included_episodes)}')
print(f'  Columns : {all_included_episodes.columns.tolist()}')
print(f'\nFirst 5 rows:')
print(all_included_episodes.head().to_string())

File 1 saved: arrhythmia_episodes_updated.csv
  Rows    : 1284
  Columns : ['caseid', 'episode_number', 'episode_start_sec', 'episode_end_sec', 'episode_duration_sec', 'episode_beat_count', 'episode_dominant_rhythm', 'episode_beat_type', 'episode_rr_cv', 'bp_source', 'nibp_outcome_imputed']

First 5 rows:
   caseid  episode_number  episode_start_sec  episode_end_sec  episode_duration_sec  episode_beat_count       episode_dominant_rhythm episode_beat_type  episode_rr_cv bp_source  nibp_outcome_imputed
0      12               1        8655.938889      8746.802778             90.863889                  15  Patterned Ventricular Ectopy                 V       0.324883   ART_MBP                 False
1      12               2        8904.375000      8918.786111             14.411111                   3                             N                 V            NaN   ART_MBP                 False
2      12               3        8971.630556      9223.980556            252.350000             

In [9]:
# ── File 2: clinical_lab_updated.csv ───────────────────────────────────────
# Keep only clinical rows for patients who still have at least one included episode.
# Adds a bp_source column so downstream code knows which BP track to load.

# Set of patient IDs that survived the episode inclusion step.
included_caseids = set(all_included_episodes['caseid'].unique())

# Filter the clinical DataFrame to those patients only.
clinical_updated = clinical[clinical['caseid'].isin(included_caseids)].copy()

# Build a caseid → bp_source lookup from the episode file.
# Each patient has episodes from only ONE source (ART_MBP OR NIBP_estimated),
# so .first() correctly picks the (only) source for that patient.
caseid_to_source = (
    all_included_episodes
    .groupby('caseid')['bp_source']
    .first()
    .to_dict()
)
clinical_updated['bp_source'] = clinical_updated['caseid'].map(caseid_to_source)

output_path_2 = OUTPUT_DIR / 'clinical_lab_updated.csv'
clinical_updated.to_csv(output_path_2, index=False)

print(f'File 2 saved: {output_path_2}')
print(f'  Rows      : {len(clinical_updated)}')
print(f'  Columns   : {len(clinical_updated.columns)} total')
print(f'  Patients excluded from original 477: {477 - len(clinical_updated)}')
print(f'    (patients whose every episode was excluded)')

File 2 saved: clinical_lab_updated.csv
  Rows      : 457
  Columns   : 104 total
  Patients excluded from original 477: 20
    (patients whose every episode was excluded)


In [10]:
# ── File 3: arrhythmia_episodes_summary.csv ────────────────────────────────
# For each episode column (excluding caseid, episode_number, bp_source,
# nibp_outcome_imputed), report the percentage of non-null and null values.
# This gives a quick overview of data completeness for the modeling columns.

EXCLUDE_EP_COLS = {'caseid', 'episode_number', 'bp_source', 'nibp_outcome_imputed'}
summary_cols    = [c for c in all_included_episodes.columns if c not in EXCLUDE_EP_COLS]

ep_summary_rows = []
n_ep_total = len(all_included_episodes)

for col in summary_cols:
    n_nonnull = int(all_included_episodes[col].notna().sum())
    n_null    = n_ep_total - n_nonnull
    ep_summary_rows.append({
        'column':       col,
        'n_total':      n_ep_total,
        'n_non_null':   n_nonnull,
        'pct_non_null': round(100 * n_nonnull / n_ep_total, 2),
        'n_null':       n_null,
        'pct_null':     round(100 * n_null    / n_ep_total, 2),
    })

ep_summary_df = pd.DataFrame(ep_summary_rows)
output_path_3 = OUTPUT_DIR / 'arrhythmia_episodes_summary.csv'
ep_summary_df.to_csv(output_path_3, index=False)

print(f'File 3 saved: {output_path_3}')
print(ep_summary_df.to_string(index=False))

File 3 saved: arrhythmia_episodes_summary.csv
                 column  n_total  n_non_null  pct_non_null  n_null  pct_null
      episode_start_sec     1284        1284        100.00       0      0.00
        episode_end_sec     1284        1284        100.00       0      0.00
   episode_duration_sec     1284        1284        100.00       0      0.00
     episode_beat_count     1284        1284        100.00       0      0.00
episode_dominant_rhythm     1284        1283         99.92       1      0.08
      episode_beat_type     1284        1156         90.03     128      9.97
          episode_rr_cv     1284         811         63.16     473     36.84


In [11]:
# ── File 4: clinical_lab_summary.csv ───────────────────────────────────────
# Same structure as File 3, but for the clinical DataFrame.
# Excludes caseid and bp_source columns.

EXCLUDE_CLIN_COLS  = {'caseid', 'bp_source'}
clinical_sum_cols  = [c for c in clinical_updated.columns if c not in EXCLUDE_CLIN_COLS]

clin_summary_rows = []
n_clin_total = len(clinical_updated)

for col in clinical_sum_cols:
    n_nonnull = int(clinical_updated[col].notna().sum())
    n_null    = n_clin_total - n_nonnull
    clin_summary_rows.append({
        'column':       col,
        'n_total':      n_clin_total,
        'n_non_null':   n_nonnull,
        'pct_non_null': round(100 * n_nonnull / n_clin_total, 2),
        'n_null':       n_null,
        'pct_null':     round(100 * n_null    / n_clin_total, 2),
    })

clin_summary_df = pd.DataFrame(clin_summary_rows)
output_path_4    = OUTPUT_DIR / 'clinical_lab_summary.csv'
clin_summary_df.to_csv(output_path_4, index=False)

print(f'File 4 saved: {output_path_4}')
print(f'(Showing first 20 rows of {len(clin_summary_df)} total columns)')
print(clin_summary_df.head(20).to_string(index=False))

File 4 saved: clinical_lab_summary.csv
(Showing first 20 rows of 102 total columns)
      column  n_total  n_non_null  pct_non_null  n_null  pct_null
   subjectid      457         457        100.00       0      0.00
   casestart      457         457        100.00       0      0.00
     caseend      457         457        100.00       0      0.00
    anestart      457         457        100.00       0      0.00
      aneend      457         457        100.00       0      0.00
     opstart      457         457        100.00       0      0.00
       opend      457         457        100.00       0      0.00
         adm      457         457        100.00       0      0.00
         dis      457         457        100.00       0      0.00
    icu_days      457         457        100.00       0      0.00
death_inhosp      457         457        100.00       0      0.00
         age      457         457        100.00       0      0.00
         sex      457         457        100.00       0   

In [12]:
# ── VERIFICATION 1: Episode counts by bp_source and nibp_outcome_imputed ───
# Expected: ART_MBP = 982, NIBP_estimated = 313 minus however many were excluded.

print('=' * 65)
print('VERIFICATION 1: Episode counts by bp_source + nibp_outcome_imputed')
print('=' * 65)

count_table = (
    all_included_episodes
    .groupby(['bp_source', 'nibp_outcome_imputed'])
    .size()
    .reset_index(name='n_episodes')
)
print(count_table.to_string(index=False))

art_count  = int((all_included_episodes['bp_source'] == 'ART_MBP').sum())
nibp_count = int((all_included_episodes['bp_source'] == 'NIBP_estimated').sum())
excl_count = len(excluded_nibp_rows)

print(f'\nART_MBP total       : {art_count}   (expected: 982)')
print(f'NIBP_estimated total: {nibp_count}   (expected: {313 - excl_count})')
print(f'NIBP episodes excl. : {excl_count}')
print(f'Grand total included: {len(all_included_episodes)}')

VERIFICATION 1: Episode counts by bp_source + nibp_outcome_imputed
     bp_source  nibp_outcome_imputed  n_episodes
       ART_MBP                 False         982
NIBP_estimated                 False         300
NIBP_estimated                  True           2

ART_MBP total       : 982   (expected: 982)
NIBP_estimated total: 302   (expected: 302)
NIBP episodes excl. : 11
Grand total included: 1284


In [13]:
# ── VERIFICATION 2: Patient counts by bp_source ────────────────────────────

print('=' * 65)
print('VERIFICATION 2: Patient counts by bp_source in clinical_lab_updated.csv')
print('=' * 65)

patient_bp_counts = clinical_updated['bp_source'].value_counts()
print(patient_bp_counts.to_string())

print(f'\nTotal patients in clinical_lab_updated : {len(clinical_updated)}')
print(f'  (477 original minus patients whose every episode was excluded)')

VERIFICATION 2: Patient counts by bp_source in clinical_lab_updated.csv
bp_source
ART_MBP           361
NIBP_estimated     96

Total patients in clinical_lab_updated : 457
  (477 original minus patients whose every episode was excluded)


In [14]:
# ── VERIFICATION 3: Raw NIBP values for 3 randomly selected NIBP episodes ──
#
# We reload the SBP/DBP waveform data for 3 episodes and print the raw values
# alongside the computed estimated MAP, confirming the formula and cleaning work.
#
# Note: NIBP cuff inflates every few minutes; between inflations the monitor
# broadcasts the same value every second. To avoid printing thousands of
# identical rows we de-duplicate, showing only rows where SBP or DBP changes
# (i.e., a new actual cuff reading has occurred).

print('=' * 65)
print('VERIFICATION 3: Raw NIBP_SBP / NIBP_DBP / estimated_MAP (3 random episodes)')
print('=' * 65)

# Select 3 random NIBP episodes reproducibly.
random.seed(42)
nibp_ep_df = all_included_episodes[
    all_included_episodes['bp_source'] == 'NIBP_estimated'
].reset_index(drop=True)

sample_idx      = random.sample(range(len(nibp_ep_df)), min(3, len(nibp_ep_df)))
sample_episodes = nibp_ep_df.iloc[sample_idx]

for _, ep_row in sample_episodes.iterrows():
    caseid  = int(ep_row['caseid'])
    ep_num  = int(ep_row['episode_number'])
    start   = int(ep_row['episode_start_sec'])
    imputed = ep_row['nibp_outcome_imputed']

    print(f'\n--- caseid={caseid}, episode={ep_num}, '
          f'episode_start={start}s, nibp_outcome_imputed={imputed} ---')

    # Reload the raw waveform data for this patient.
    arr2 = vitaldb.load_case(caseid, [NIBP_SBP_TRACK, NIBP_DBP_TRACK], INTERVAL_SEC)
    sbp2_raw = arr2[:, 0].copy()   # raw SBP before any cleaning
    dbp2_raw = arr2[:, 1].copy()   # raw DBP before any cleaning
    n2       = len(sbp2_raw)

    # Apply the same artifact cleaning as the main loop.
    sbp2 = sbp2_raw.copy()
    dbp2 = dbp2_raw.copy()
    sbp2[(sbp2 < SBP_MIN) | (sbp2 > SBP_MAX)] = np.nan
    dbp2[(dbp2 < DBP_MIN) | (dbp2 > DBP_MAX)] = np.nan
    bad2 = sbp2 <= dbp2
    sbp2[bad2] = np.nan
    dbp2[bad2] = np.nan
    est_map2 = dbp2 + (sbp2 - dbp2) / 3
    est_map2[(est_map2 < MAP_MIN) | (est_map2 > MAP_MAX)] = np.nan

    # Determine which window to display.
    if imputed:
        disp_start = max(0, start - EXPANDED_BEFORE_SEC)
        disp_end   = min(start + EXPANDED_AFTER_SEC, n2)
        print(f'  [Expanded window shown: {disp_start}s to {disp_end}s]')
    else:
        disp_start = max(0, start)
        disp_end   = min(start + OUTCOME_WINDOW_SEC, n2)
        print(f'  [Primary 5-min window shown: {disp_start}s to {disp_end}s]')

    # Collect de-duplicated rows (only print when SBP or DBP value changes).
    rows_to_show = []
    prev_sbp, prev_dbp = None, None

    for t in range(disp_start, disp_end):
        s_raw = sbp2_raw[t]
        d_raw = dbp2_raw[t]
        m_val = est_map2[t]

        # Only include this second if at least one raw value is non-NaN AND has changed.
        if not (np.isnan(s_raw) and np.isnan(d_raw)):
            if s_raw != prev_sbp or d_raw != prev_dbp:
                rows_to_show.append({
                    'time_s':        t,
                    'NIBP_SBP_raw':  round(float(s_raw), 1) if not np.isnan(s_raw) else None,
                    'NIBP_DBP_raw':  round(float(d_raw), 1) if not np.isnan(d_raw) else None,
                    'estimated_MAP': round(float(m_val), 1) if not np.isnan(m_val) else None,
                    'MAP_valid':     not np.isnan(m_val),
                })
                prev_sbp, prev_dbp = s_raw, d_raw

    if rows_to_show:
        print(pd.DataFrame(rows_to_show).to_string(index=False))
    else:
        print('  (no non-NaN NIBP readings found in this window)')

VERIFICATION 3: Raw NIBP_SBP / NIBP_DBP / estimated_MAP (3 random episodes)

--- caseid=1276, episode=2, episode_start=3001s, nibp_outcome_imputed=False ---
  [Primary 5-min window shown: 3001s to 3301s]
 time_s  NIBP_SBP_raw  NIBP_DBP_raw  estimated_MAP  MAP_valid
   3002         167.0          99.0          121.7       True
   3100         155.0          91.0          112.3       True
   3236         146.0          92.0          110.0       True

--- caseid=285, episode=3, episode_start=414s, nibp_outcome_imputed=False ---
  [Primary 5-min window shown: 414s to 714s]
 time_s  NIBP_SBP_raw  NIBP_DBP_raw  estimated_MAP  MAP_valid
    415         118.0          58.0           78.0       True
    441         102.0          54.0           70.0       True
    507          92.0          50.0           64.0       True
    573          94.0          49.0           64.0       True
    625          84.0          44.0           57.3       True
    683         103.0          57.0           72.3  

In [ ]:
# ── VERIFICATION 4: bp_source consistency between both output files ─────────
#
# Rule: every patient labeled NIBP_estimated in clinical_lab_updated.csv must
# have ONLY NIBP_estimated episodes in arrhythmia_episodes_updated.csv, and
# every ART_MBP patient must have ONLY ART_MBP episodes.

print('=' * 65)
print('VERIFICATION 4: bp_source consistency between output files')
print('=' * 65)

passed = True

# --- Check NIBP patients ---
nibp_caseids_clinical = set(
    clinical_updated.loc[clinical_updated['bp_source'] == 'NIBP_estimated', 'caseid']
)
episodes_of_nibp_patients = all_included_episodes[
    all_included_episodes['caseid'].isin(nibp_caseids_clinical)
]
nibp_mismatched = episodes_of_nibp_patients[
    episodes_of_nibp_patients['bp_source'] != 'NIBP_estimated'
]
if len(nibp_mismatched) == 0:
    print('PASS: All NIBP_estimated patients have only NIBP_estimated episodes.')
else:
    print(f'FAIL: {len(nibp_mismatched)} episode(s) under NIBP patients have wrong bp_source:')
    print(nibp_mismatched[['caseid', 'episode_number', 'bp_source']].to_string(index=False))
    passed = False

# --- Check ART_MBP patients ---
art_caseids_clinical = set(
    clinical_updated.loc[clinical_updated['bp_source'] == 'ART_MBP', 'caseid']
)
episodes_of_art_patients = all_included_episodes[
    all_included_episodes['caseid'].isin(art_caseids_clinical)
]
art_mismatched = episodes_of_art_patients[
    episodes_of_art_patients['bp_source'] != 'ART_MBP'
]
if len(art_mismatched) == 0:
    print('PASS: All ART_MBP patients have only ART_MBP episodes.')
else:
    print(f'FAIL: {len(art_mismatched)} episode(s) under ART_MBP patients have wrong bp_source:')
    print(art_mismatched[['caseid', 'episode_number', 'bp_source']].to_string(index=False))
    passed = False

print()
if passed:
    print('All verification checks PASSED. Files are consistent and ready for Stage 3.')
else:
    print('One or more verification checks FAILED. Review the output above.')